In [1]:
# 06_decision_analysis.ipynb
# =============================================================================
# From prediction to DECISION value. A credit screen does not rank for its own
# sake; it supports a lend / decline decision under asymmetric costs. We translate
# the relational (contagion) signal into decision terms and compare it against
# the intra-only model on the strict split (mask A):
#   (1) CALIBRATION: reliability curves and Brier scores (are the predicted
#       probabilities usable as probabilities, not just ranks?).
#   (2) COST-SENSITIVE EXPECTED COST: with a cost ratio C_FN : C_FP for missing a
#       bankruptcy vs a false alarm, sweep the decision threshold and report the
#       minimum achievable expected cost per company for each model.
#   (3) DECISION CURVE ANALYSIS (net benefit) across threshold probabilities,
#       against treat-all / treat-none references.
# Probabilities are averaged over 5 seeds for stability. Figures grayscale,
# dpi 600, png + pdf, no caption.
# =============================================================================
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from torch_geometric.nn import HeteroConv, SAGEConv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[0]
PROCESSED = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.size": 10})
sns.set_theme(style="whitegrid", palette="Greys_r")
SEEDS = list(range(5))
COST_RATIO = 5.0   # missing a bankruptcy costs 5x a false alarm (illustrative)


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)


data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
Xc = comp.x.numpy().astype(np.float64)
Xc[:, :2] = np.log1p(np.clip(Xc[:, :2], 0, None))
Xc = StandardScaler().fit(Xc[comp.train_mask.numpy()]).transform(Xc).astype(np.float32)
comp.x = torch.tensor(Xc)
y = comp.y
trm, vam, tem = comp.train_mask, comp.val_mask, comp.test_mask
EDGE_TYPES = [et for et in data.edge_types if int(et[1].split("_")[1]) <= 11]
ei_dict = {et: data[et].edge_index for et in EDGE_TYPES}
IN_DIMS = {"company": comp.x.size(1), "person": data["person"].x.size(1)}


class HeteroGNN(nn.Module):
    def __init__(self, hidden=64, out=2, layers=2, dropout=0.3):
        super().__init__()
        self.drop = dropout
        self.lin_in = nn.ModuleDict({nt: nn.Linear(d, hidden) for nt, d in IN_DIMS.items()})
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv((hidden, hidden), hidden) for et in EDGE_TYPES},
                       aggr="sum") for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out))

    def forward(self, xd, eid):
        h = {nt: F.relu(self.lin_in[nt](x)) for nt, x in xd.items()}
        for conv in self.convs:
            h = conv(h, eid)
            h = {nt: F.dropout(F.relu(v), p=self.drop, training=self.training)
                 for nt, v in h.items()}
        return self.head(h["company"])


class IntraMLP(nn.Module):
    def __init__(self, d, hidden=64, out=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, out))

    def forward(self, x):
        return self.net(x)


def class_w():
    cls = torch.bincount(y[trm], minlength=2).float()
    return cls.sum() / (2 * cls.clamp(min=1))


def train_probs(kind, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    w = class_w()
    if kind == "contagion":
        model = HeteroGNN(); xd = {nt: data[nt].x for nt in data.node_types}
        fwd = lambda: model(xd, ei_dict)
    else:
        model = IntraMLP(comp.x.size(1)); fwd = lambda: model(comp.x)
    opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=5e-4)
    best_val, best_state, wait = -1, None, 0
    for _ in range(200):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(fwd()[trm], y[trm], weight=w)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = F.softmax(fwd(), 1)[:, 1].numpy()
        va = roc_auc_score(y[vam].numpy(), p[vam.numpy()])
        if va > best_val:
            best_val, best_state, wait = va, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 30:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        return F.softmax(fwd(), 1)[:, 1].numpy()[tem.numpy()]


yt = y[tem.numpy()].numpy()
probs = {}
for kind in ["intra", "contagion"]:
    ps = np.mean([train_probs(kind, s) for s in SEEDS], axis=0)
    probs[kind] = ps
    print(f"{kind:9s}: test AUC {roc_auc_score(yt, ps):.3f}  Brier {brier_score_loss(yt, ps):.4f}")

# --- 1. Calibration (reliability curve, 10 bins) -----------------------------
def reliability(y_true, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    xs, ys, ns = [], [], []
    for b in range(bins):
        m = idx == b
        if m.sum() > 0:
            xs.append(p[m].mean()); ys.append(y_true[m].mean()); ns.append(int(m.sum()))
    return np.array(xs), np.array(ys), np.array(ns)

cal_rows = []
fig, ax = plt.subplots(figsize=(4.2, 4.0))
ax.plot([0, 1], [0, 1], color="black", linewidth=0.8, linestyle="--")
for kind, mk, col in [("intra", "s", "#999999"), ("contagion", "o", "#222222")]:
    xs, ys, ns = reliability(yt, probs[kind])
    ax.plot(xs, ys, marker=mk, color=col, linewidth=1.2, markersize=5,
            label=f"{kind} (Brier {brier_score_loss(yt, probs[kind]):.3f})")
    for a_, b_, n_ in zip(xs, ys, ns):
        cal_rows.append({"model": kind, "pred_mean": round(a_, 4),
                         "obs_rate": round(b_, 4), "n": n_})
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed bankruptcy rate")
ax.legend(frameon=True, fontsize=8, loc="upper left")
savefig(fig, "fig_calibration")
pd.DataFrame(cal_rows).to_csv(TAB / "06_calibration.csv", index=False)

# --- 2. Cost-sensitive expected cost vs threshold ----------------------------
thr = np.linspace(0.01, 0.99, 99)
def expected_cost(p, c_ratio):
    out = []
    for t in thr:
        pred = (p >= t).astype(int)
        fn = int(((pred == 0) & (yt == 1)).sum())
        fp = int(((pred == 1) & (yt == 0)).sum())
        out.append((c_ratio * fn + fp) / len(yt))
    return np.array(out)

cost_rows = []
fig, ax = plt.subplots(figsize=(5.0, 3.6))
for kind, col in [("intra", "#999999"), ("contagion", "#222222")]:
    ec = expected_cost(probs[kind], COST_RATIO)
    ax.plot(thr, ec, color=col, linewidth=1.4, label=kind)
    j = int(np.argmin(ec))
    ax.scatter([thr[j]], [ec[j]], color=col, s=25, zorder=5)
    cost_rows.append({"model": kind, "cost_ratio": COST_RATIO,
                      "min_expected_cost": round(float(ec[j]), 4),
                      "best_threshold": round(float(thr[j]), 3)})
ax.set_xlabel("Decision threshold"); ax.set_ylabel(f"Expected cost per company (C_FN:C_FP={COST_RATIO:.0f}:1)")
ax.legend(frameon=True, fontsize=8)
savefig(fig, "fig_cost_curve")
cost_df = pd.DataFrame(cost_rows); cost_df.to_csv(TAB / "06_expected_cost.csv", index=False)
print("\n=== Minimum expected cost (C_FN:C_FP = 5:1) ===")
print(cost_df.to_string(index=False))
saving = cost_df.set_index("model").loc["intra", "min_expected_cost"] - \
         cost_df.set_index("model").loc["contagion", "min_expected_cost"]
print(f"contagion lowers minimum expected cost by {saving:.4f} per company "
      f"({100*saving/cost_df.set_index('model').loc['intra','min_expected_cost']:.1f}% relative)")

# --- 3. Decision curve analysis (net benefit) --------------------------------
pt = np.linspace(0.01, 0.9, 90)
prev = yt.mean()
def net_benefit(p):
    nb = []
    for t in pt:
        pred = (p >= t).astype(int)
        tp = int(((pred == 1) & (yt == 1)).sum())
        fp = int(((pred == 1) & (yt == 0)).sum())
        nb.append(tp / len(yt) - fp / len(yt) * (t / (1 - t)))
    return np.array(nb)

nb_all = prev - (1 - prev) * (pt / (1 - pt))
fig, ax = plt.subplots(figsize=(5.0, 3.6))
ax.plot(pt, net_benefit(probs["contagion"]), color="#222222", linewidth=1.5, label="Contagion")
ax.plot(pt, net_benefit(probs["intra"]), color="#999999", linewidth=1.5, label="Intra-only")
ax.plot(pt, nb_all, color="black", linestyle="--", linewidth=0.8, label="Screen all")
ax.axhline(0, color="black", linestyle=":", linewidth=0.8)
ax.set_xlabel("Threshold probability"); ax.set_ylabel("Net benefit")
ax.set_ylim(-0.1, prev + 0.05)
ax.legend(frameon=True, fontsize=8)
savefig(fig, "fig_decision_curve")
nb_df = pd.DataFrame({"threshold_prob": pt.round(3),
                      "nb_contagion": net_benefit(probs["contagion"]).round(4),
                      "nb_intra": net_benefit(probs["intra"]).round(4),
                      "nb_screen_all": nb_all.round(4)})
nb_df.to_csv(TAB / "06_decision_curve.csv", index=False)
print("\n06_decision_analysis complete: calibration + cost curve + decision curve; "
      "3 figures, 3 tables.")


intra    : test AUC 0.687  Brier 0.2472


contagion: test AUC 0.780  Brier 0.2109



=== Minimum expected cost (C_FN:C_FP = 5:1) ===
    model  cost_ratio  min_expected_cost  best_threshold
    intra         5.0             0.3442            0.02
contagion         5.0             0.3360            0.03
contagion lowers minimum expected cost by 0.0082 per company (2.4% relative)



06_decision_analysis complete: calibration + cost curve + decision curve; 3 figures, 3 tables.
